# CS4063 NLP Project
##  Sarcasm Detection in Tweets using Transformer Models




In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 ─ Install dependencies
# ═══════════════════════════════════════════════════════════════════
!pip install transformers datasets emoji matplotlib scikit-learn -q

import os, json, re, csv, random, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

try:
    import emoji as emoji_lib
    EMOJI_OK = True
except ImportError:
    EMOJI_OK = False

warnings.filterwarnings('ignore')
os.makedirs('results/plots', exist_ok=True)

# ── GPU check ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠  No GPU detected!')
    print('  → Runtime → Change runtime type → T4 GPU → Save, then re-run')

print('✓ All libraries loaded successfully')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.7 MB/s eta 0:00:00
✓ GPU: Tesla T4
✓ All libraries loaded successfully


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 ─ Download SemEval-2018 Task 3 dataset
# ═══════════════════════════════════════════════════════════════════
!git clone https://github.com/Cyvhee/SemEval2018-Task3.git -q

os.makedirs('data/processed', exist_ok=True)
random.seed(42)

# ── Load training tweets ──────────────────────────────────────────
samples = []
train_file = 'SemEval2018-Task3/datasets/train/SemEval2018-T3-train-taskA.txt'
with open(train_file, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split('\t')
        if len(parts) >= 3:
            try:
                samples.append({'text': parts[2], 'label': int(parts[1])})
            except:
                continue

# ── Load official test set (gold labels) ─────────────────────────
test_file = 'SemEval2018-Task3/datasets/goldtest_TaskA/SemEval2018-T3_gold_test_taskA_emoji.txt'
test_raw = []
if os.path.exists(test_file):
    with open(test_file, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    test_raw.append({'text': parts[2], 'label': int(parts[1])})
                except:
                    continue

# ── Split ─────────────────────────────────────────────────────────
random.shuffle(samples)
n = len(samples)
if test_raw:
    cut = int(n * 0.9)
    TRAIN_DATA_RAW = samples[:cut]
    VAL_DATA_RAW   = samples[cut:]
    TEST_DATA_RAW  = test_raw
else:
    TRAIN_DATA_RAW = samples[:int(n * 0.8)]
    VAL_DATA_RAW   = samples[int(n * 0.8):int(n * 0.9)]
    TEST_DATA_RAW  = samples[int(n * 0.9):]

n_sarc = sum(1 for s in TRAIN_DATA_RAW if s['label'] == 1)
print(f'✓ Dataset loaded')
print(f'  Train: {len(TRAIN_DATA_RAW)} | Val: {len(VAL_DATA_RAW)} | Test: {len(TEST_DATA_RAW)}')
print(f'  Sarcastic in train: {n_sarc}/{len(TRAIN_DATA_RAW)} ({100*n_sarc/len(TRAIN_DATA_RAW):.1f}%)')
print(f'  Non-sarcastic:      {len(TRAIN_DATA_RAW)-n_sarc}/{len(TRAIN_DATA_RAW)} ({100*(len(TRAIN_DATA_RAW)-n_sarc)/len(TRAIN_DATA_RAW):.1f}%)')
print(f'  Sample tweet: "{TRAIN_DATA_RAW[0]["text"][:80]}"')

✓ Dataset loaded
  Train: 3450 | Val: 384 | Test: 784
  Sarcastic in train: 1728/3450 (50.1%)
  Non-sarcastic:      1722/3450 (49.9%)
  Sample tweet: "Ya and listening to them talk about how hot girls are is a favorite past time. -"


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 ─ Tweet preprocessing pipeline
# ═══════════════════════════════════════════════════════════════════
URL_PAT     = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
MENTION_PAT = re.compile(r'@\w+')
HASHTAG_PAT = re.compile(r'#(\w+)')
CAMEL_PAT   = re.compile(r'([a-z])([A-Z])')
REPEAT_PAT  = re.compile(r'(.)\1{3,}')
SPACE_PAT   = re.compile(r'\s+')
RT_PAT      = re.compile(r'^RT\s*:?\s*', re.IGNORECASE)

def clean_tweet(text, mode='bertweet'):
    """
    Clean a single tweet for transformer input.
    mode='bertweet' → @USER and HTTPURL tokens (matches BERTweet pre-training)
    mode='standard' → removes mentions and URLs
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    text = RT_PAT.sub('', text)
    text = URL_PAT.sub('HTTPURL' if mode == 'bertweet' else '', text)
    text = MENTION_PAT.sub('@USER' if mode == 'bertweet' else '', text)
    # Hashtag segmentation: #GreatPlan → Great Plan
    def segment(m):
        tag = CAMEL_PAT.sub(r'\1 \2', m.group(1)).replace('_', ' ')
        return tag.strip()
    text = HASHTAG_PAT.sub(segment, text)
    # Emoji → text description
    if EMOJI_OK:
        text = emoji_lib.demojize(text, delimiters=(' ', ' '))
    else:
        text = text.encode('ascii', 'ignore').decode('ascii')
    text = REPEAT_PAT.sub(r'\1\1', text)  # sooooo → soo
    text = SPACE_PAT.sub(' ', text)
    return text.strip()

def preprocess_split(samples, mode='bertweet'):
    cleaned = [{'text': clean_tweet(s['text'], mode), 'label': s['label']}
               for s in samples if clean_tweet(s['text'], mode)]
    return cleaned

TRAIN_DATA = preprocess_split(TRAIN_DATA_RAW)
VAL_DATA   = preprocess_split(VAL_DATA_RAW)
TEST_DATA  = preprocess_split(TEST_DATA_RAW)

print(f'✓ Preprocessing complete')
print(f'  After cleaning — Train: {len(TRAIN_DATA)} | Val: {len(VAL_DATA)} | Test: {len(TEST_DATA)}')
print(f'\nSample cleaned tweets:')
for s in TRAIN_DATA[:4]:
    tag = '[SARCASTIC]' if s['label'] == 1 else '[LITERAL]  '
    print(f'  {tag} {s["text"][:90]}')

✓ Preprocessing complete
  After cleaning — Train: 3450 | Val: 384 | Test: 784

Sample cleaned tweets:
  [SARCASTIC] Ya and listening to them talk about how hot girls are is a favorite past time. -_-
  [SARCASTIC] White people are irrelevant at this point. The black community needs to rise ABOVE y'all a
  [LITERAL]   @USER I just did it. What's the prize?
  [LITERAL]   What an awesome performance by the Choral students @USER tonight! They're developing young


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 ─ Weak baselines
# ═══════════════════════════════════════════════════════════════════
def to_xy(samples):
    return [s['text'] for s in samples], [s['label'] for s in samples]

def get_metrics(true_labels, pred_labels):
    acc = accuracy_score(true_labels, pred_labels)
    p, r, f1, _ = precision_recall_fscore_support(
        true_labels, pred_labels, average='binary', pos_label=1, zero_division=0
    )
    return {
        'accuracy':  round(float(acc), 4),
        'precision': round(float(p),   4),
        'recall':    round(float(r),   4),
        'f1':        round(float(f1),  4),
    }

tr_x, tr_y = to_xy(TRAIN_DATA)
va_x, va_y = to_xy(VAL_DATA)
te_x, te_y = to_xy(TEST_DATA)

BASELINE_RESULTS = []

# ── 1. Majority class ─────────────────────────────────────────────
clf = DummyClassifier(strategy='most_frequent')
clf.fit(tr_x, tr_y)
m = get_metrics(te_y, clf.predict(te_x))
BASELINE_RESULTS.append({'name': 'Majority Class', **m})
print(f'Majority Class        → F1: {m["f1"]:.4f}  Acc: {m["accuracy"]:.4f}')

# ── 2. TF-IDF + Logistic Regression ──────────────────────────────
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                               sublinear_tf=True, min_df=2)),
    ('clf',  LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                                 solver='lbfgs', random_state=42)),
])
lr_pipe.fit(tr_x, tr_y)
m = get_metrics(te_y, lr_pipe.predict(te_x))
BASELINE_RESULTS.append({'name': 'TF-IDF + Logistic Regression', **m})
print(f'TF-IDF + LogReg       → F1: {m["f1"]:.4f}  Acc: {m["accuracy"]:.4f}')

# ── 3. TF-IDF + LinearSVC ─────────────────────────────────────────
svm_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000,
                               sublinear_tf=True, min_df=2)),
    ('clf',  LinearSVC(C=0.5, max_iter=2000, class_weight='balanced', random_state=42)),
])
svm_pipe.fit(tr_x, tr_y)
m = get_metrics(te_y, svm_pipe.predict(te_x))
BASELINE_RESULTS.append({'name': 'TF-IDF + LinearSVC', **m})
print(f'TF-IDF + LinearSVC    → F1: {m["f1"]:.4f}  Acc: {m["accuracy"]:.4f}')

print('\n✓ Baselines complete')

Majority Class        → F1: 0.5680  Acc: 0.3967
TF-IDF + LogReg       → F1: 0.5926  Acc: 0.6633
TF-IDF + LinearSVC    → F1: 0.5706  Acc: 0.6429

✓ Baselines complete


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 ─ Training infrastructure
# ═══════════════════════════════════════════════════════════════════
MODEL_REGISTRY = {
    'bert':     'bert-base-uncased',
    'roberta':  'roberta-base',
    'bertweet': 'vinai/bertweet-base',
}

ALL_RESULTS = []  # collects every run for the final table

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SarcasmDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        enc  = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(item['label'], dtype=torch.long),
        }


def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['label'].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=labs)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += out.loss.item()
        preds_all.extend(torch.argmax(out.logits, -1).cpu().numpy())
        labels_all.extend(labs.cpu().numpy())
    return total_loss / len(loader), get_metrics(labels_all, preds_all)


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['label'].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=labs)
        total_loss += out.loss.item()
        preds_all.extend(torch.argmax(out.logits, -1).cpu().numpy())
        labels_all.extend(labs.cpu().numpy())
    return total_loss / len(loader), get_metrics(labels_all, preds_all), labels_all, preds_all


def fine_tune(model_key, epochs=5, batch_size=32, lr=2e-5,
              freeze_encoder=False, label=None):
    """
    Fine-tune a transformer model and record results.
    model_key : 'bert' | 'roberta' | 'bertweet'
    """
    set_seed(42)
    model_name = MODEL_REGISTRY[model_key]
    run_label  = label or (model_name.split('/')[-1] + (' (frozen)' if freeze_encoder else ''))

    print(f'\n{"═"*62}')
    print(f'  Training : {run_label}')
    print(f'  Model    : {model_name}')
    print(f'  Epochs   : {epochs}  |  Batch: {batch_size}  |  LR: {lr}')
    if freeze_encoder:
        print(f'  Mode     : frozen encoder — classifier head only')
    print(f'{"═"*62}')

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model     = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True
    ).to(device)

    if freeze_encoder:
        for name, param in model.named_parameters():
            if not any(k in name for k in ['classifier', 'pooler']):
                param.requires_grad = False
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable parameters: {n_trainable:,}')

    pin = (device.type == 'cuda')
    train_loader = DataLoader(SarcasmDataset(TRAIN_DATA, tokenizer),
                              batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=pin)
    val_loader   = DataLoader(SarcasmDataset(VAL_DATA,   tokenizer),
                              batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    test_loader  = DataLoader(SarcasmDataset(TEST_DATA,  tokenizer),
                              batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)

    total_steps = len(train_loader) * epochs
    optimizer   = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01,
    )
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )

    best_val_f1  = 0.0
    best_state   = None
    training_log = []

    for epoch in range(1, epochs + 1):
        tr_loss, tr_m       = train_one_epoch(model, train_loader, optimizer, scheduler)
        vl_loss, vl_m, _, _ = evaluate_model(model, val_loader)
        training_log.append({
            'epoch': epoch,
            'train_loss': round(tr_loss, 4), 'val_loss': round(vl_loss, 4),
            **{f'train_{k}': v for k, v in tr_m.items()},
            **{f'val_{k}':   v for k, v in vl_m.items()},
        })
        print(f'  Ep {epoch}/{epochs} — '
              f'tr_loss: {tr_loss:.4f}  tr_f1: {tr_m["f1"]:.4f}  '
              f'vl_loss: {vl_loss:.4f}  vl_f1: {vl_m["f1"]:.4f}')
        if vl_m['f1'] > best_val_f1:
            best_val_f1 = vl_m['f1']
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'  ✓ Best val F1: {best_val_f1:.4f}')

    # ── Evaluate best checkpoint on test set ──
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    te_loss, te_m, te_true, te_preds = evaluate_model(model, test_loader)

    print(f'\n  TEST → Acc: {te_m["accuracy"]:.4f}  '
          f'P: {te_m["precision"]:.4f}  R: {te_m["recall"]:.4f}  F1: {te_m["f1"]:.4f}')

    result = {
        'name':           run_label,
        'model_key':      model_key,
        'model_name':     model_name,
        'freeze_encoder': freeze_encoder,
        'best_val_f1':    best_val_f1,
        'test_metrics':   te_m,
        'test_loss':      round(te_loss, 4),
        'training_log':   training_log,
        'confusion_matrix': confusion_matrix(te_true, te_preds).tolist(),
        'test_true':      [int(x) for x in te_true],
        'test_preds':     [int(x) for x in te_preds],
        'test_texts':     [s['text'] for s in TEST_DATA],
    }
    ALL_RESULTS.append(result)
    return result


print('✓ Training infrastructure ready — run Cells 6–9 to start training')

✓ Training infrastructure ready — run Cells 6–9 to start training


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 ─ Fine-tune BERT-base  (~15 min)
# ═══════════════════════════════════════════════════════════════════
bert_result = fine_tune('bert', epochs=5, batch_size=32, lr=2e-5)


══════════════════════════════════════════════════════════════
  Training : bert-base-uncased
  Model    : bert-base-uncased
  Epochs   : 5  |  Batch: 32  |  LR: 2e-05
══════════════════════════════════════════════════════════════


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Ep 1/5 — tr_loss: 0.6682  tr_f1: 0.5875  vl_loss: 0.6084  vl_f1: 0.6576
  ✓ Best val F1: 0.6576
  Ep 2/5 — tr_loss: 0.5849  tr_f1: 0.6802  vl_loss: 0.5715  vl_f1: 0.6648
  ✓ Best val F1: 0.6648
  Ep 3/5 — tr_loss: 0.4722  tr_f1: 0.7737  vl_loss: 0.5874  vl_f1: 0.6649
  ✓ Best val F1: 0.6649
  Ep 4/5 — tr_loss: 0.3762  tr_f1: 0.8374  vl_loss: 0.6930  vl_f1: 0.7050
  ✓ Best val F1: 0.7050
  Ep 5/5 — tr_loss: 0.2922  tr_f1: 0.8857  vl_loss: 0.6650  vl_f1: 0.6880

  TEST → Acc: 0.6505  P: 0.5433  R: 0.7460  F1: 0.6287


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 ─ Fine-tune RoBERTa-base  (~15 min)
# ═══════════════════════════════════════════════════════════════════
roberta_result = fine_tune('roberta', epochs=3, batch_size=32, lr=2e-5)


══════════════════════════════════════════════════════════════
  Training : roberta-base
  Model    : roberta-base
  Epochs   : 3  |  Batch: 32  |  LR: 2e-05
══════════════════════════════════════════════════════════════


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Ep 1/3 — tr_loss: 0.6717  tr_f1: 0.5017  vl_loss: 0.5927  vl_f1: 0.6983
  ✓ Best val F1: 0.6983
  Ep 2/3 — tr_loss: 0.5500  tr_f1: 0.7041  vl_loss: 0.5763  vl_f1: 0.7015
  ✓ Best val F1: 0.7015
  Ep 3/3 — tr_loss: 0.4357  tr_f1: 0.7950  vl_loss: 0.5768  vl_f1: 0.7147
  ✓ Best val F1: 0.7147

  TEST → Acc: 0.7564  P: 0.6523  R: 0.8264  F1: 0.7291


In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 ─ Fine-tune BERTweet — MAIN MODEL  (~20 min)
# ═══════════════════════════════════════════════════════════════════
bertweet_result = fine_tune('bertweet', epochs=5, batch_size=32, lr=2e-5)


══════════════════════════════════════════════════════════════
  Training : bertweet-base
  Model    : vinai/bertweet-base
  Epochs   : 5  |  Batch: 32  |  LR: 2e-05
══════════════════════════════════════════════════════════════


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

  Ep 1/5 — tr_loss: 0.6580  tr_f1: 0.5721  vl_loss: 0.5361  vl_f1: 0.7049
  ✓ Best val F1: 0.7049
  Ep 2/5 — tr_loss: 0.4986  tr_f1: 0.7591  vl_loss: 0.5143  vl_f1: 0.7500
  ✓ Best val F1: 0.7500
  Ep 3/5 — tr_loss: 0.3637  tr_f1: 0.8468  vl_loss: 0.5351  vl_f1: 0.7443
  Ep 4/5 — tr_loss: 0.2511  tr_f1: 0.9097  vl_loss: 0.5843  vl_f1: 0.7592
  ✓ Best val F1: 0.7592
  Ep 5/5 — tr_loss: 0.1843  tr_f1: 0.9404  vl_loss: 0.6210  vl_f1: 0.7763
  ✓ Best val F1: 0.7763

  TEST → Acc: 0.8278  P: 0.7304  R: 0.8971  F1: 0.8052


In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 ─ Ablation: BERTweet frozen encoder  (~10 min)
# ═══════════════════════════════════════════════════════════════════
bertweet_frozen = fine_tune(
    'bertweet', epochs=5, batch_size=32, lr=2e-5,
    freeze_encoder=True, label='bertweet-base (frozen)'
)


══════════════════════════════════════════════════════════════
  Training : bertweet-base (frozen)
  Model    : vinai/bertweet-base
  Epochs   : 5  |  Batch: 32  |  LR: 2e-05
  Mode     : frozen encoder — classifier head only
══════════════════════════════════════════════════════════════


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Trainable parameters: 592,130
  Ep 1/5 — tr_loss: 0.6939  tr_f1: 0.4993  vl_loss: 0.6889  vl_f1: 0.4969
  ✓ Best val F1: 0.4969
  Ep 2/5 — tr_loss: 0.6894  tr_f1: 0.5283  vl_loss: 0.6847  vl_f1: 0.5938
  ✓ Best val F1: 0.5938
  Ep 3/5 — tr_loss: 0.6866  tr_f1: 0.5630  vl_loss: 0.6821  vl_f1: 0.6027
  ✓ Best val F1: 0.6027
  Ep 4/5 — tr_loss: 0.6828  tr_f1: 0.5527  vl_loss: 0.6819  vl_f1: 0.6409
  ✓ Best val F1: 0.6409
  Ep 5/5 — tr_loss: 0.6839  tr_f1: 0.5749  vl_loss: 0.6803  vl_f1: 0.6308

  TEST → Acc: 0.5077  P: 0.4352  R: 0.8103  F1: 0.5663


In [10]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 ─ Error analysis + all plots
# ═══════════════════════════════════════════════════════════════════

# ── Pick best model ───────────────────────────────────────────────
best_run = max(ALL_RESULTS, key=lambda r: r['test_metrics']['f1'])
print(f'Best model : {best_run["name"]}')
print(f'Test F1    : {best_run["test_metrics"]["f1"]:.4f}')
print(f'Test Acc   : {best_run["test_metrics"]["accuracy"]:.4f}')

true_labels  = best_run['test_true']
pred_labels  = best_run['test_preds']
texts        = best_run['test_texts']

fp_samples = [(texts[i], true_labels[i], pred_labels[i])
              for i in range(len(true_labels)) if true_labels[i]==0 and pred_labels[i]==1]
fn_samples = [(texts[i], true_labels[i], pred_labels[i])
              for i in range(len(true_labels)) if true_labels[i]==1 and pred_labels[i]==0]
tp_samples = [(texts[i], true_labels[i], pred_labels[i])
              for i in range(len(true_labels)) if true_labels[i]==1 and pred_labels[i]==1]

# ── Error analysis printout ───────────────────────────────────────
print(f'\n{"─"*65}')
print(f'ERROR ANALYSIS — {best_run["name"]}')
print(f'{"─"*65}')
print(f'Total test samples : {len(true_labels)}')
print(f'True Positives     : {len(tp_samples)}')
print(f'False Positives    : {len(fp_samples)}  (literal → predicted sarcastic)')
print(f'False Negatives    : {len(fn_samples)}  (sarcastic → predicted literal)')

print(f'\n── False Positives (literal tweets called sarcastic) ─────────')
for i, (t, _, _) in enumerate(fp_samples[:10], 1):
    print(f'  [{i:2d}] {t[:100]}')

print(f'\n── False Negatives (sarcastic tweets the model missed) ───────')
for i, (t, _, _) in enumerate(fn_samples[:10], 1):
    print(f'  [{i:2d}] {t[:100]}')

# ── Pattern analysis ──────────────────────────────────────────────
indicators = {
    'obviously / clearly':  r'\b(obviously|clearly|of course)\b',
    'totally / definitely': r'\b(totally|definitely|absolutely)\b',
    'wow / oh great':       r'\b(wow|oh great|great job|oh sure)\b',
    'question marks':       r'\?',
    'exclamation marks':    r'!',
    'uppercase words':      r'\b[A-Z]{3,}\b',
    'yeah / sure':          r'\b(yeah|sure|right)\b',
}
print(f'\n── Pattern frequency in False Negatives ──────────────────────')
for name, pat in indicators.items():
    c = sum(1 for t,_,_ in fn_samples if re.search(pat, t, re.IGNORECASE))
    if c and len(fn_samples):
        print(f'  {name:<30}: {c}/{len(fn_samples)} ({100*c/len(fn_samples):.0f}%)')

# ── Classification report ─────────────────────────────────────────
print(f'\n── Full classification report ────────────────────────────────')
print(classification_report(true_labels, pred_labels,
                             target_names=['Not Sarcastic','Sarcastic'], digits=4))

# ══ PLOT 1: Confusion matrix ══════════════════════════════════════
cm = np.array(best_run['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Not Sarcastic','Sarcastic'], fontsize=11)
ax.set_yticklabels(['Not Sarcastic','Sarcastic'], fontsize=11)
ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix — {best_run["name"]}', fontsize=12)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                fontsize=16, fontweight='bold',
                color='white' if cm[i,j] > thresh else 'black')
plt.tight_layout()
plt.savefig('results/plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Confusion matrix saved')

# ══ PLOT 2: Model comparison bar chart ═══════════════════════════
all_rows = BASELINE_RESULTS + [{'name': r['name'], **r['test_metrics']} for r in ALL_RESULTS]
names = [r['name'].replace('vinai/', '').replace('-base','') for r in all_rows]
f1s   = [r['f1']        for r in all_rows]
precs = [r['precision'] for r in all_rows]
recs  = [r['recall']    for r in all_rows]
accs  = [r['accuracy']  for r in all_rows]

x = np.arange(len(names)); w = 0.2
fig, ax = plt.subplots(figsize=(max(9, len(names)*1.8), 5))
b1 = ax.bar(x-1.5*w, f1s,   w, label='F1',        color='#2E5FA3', alpha=0.88)
b2 = ax.bar(x-0.5*w, precs, w, label='Precision',  color='#2DA39C', alpha=0.88)
b3 = ax.bar(x+0.5*w, recs,  w, label='Recall',     color='#E0783C', alpha=0.88)
b4 = ax.bar(x+1.5*w, accs,  w, label='Accuracy',   color='#7B4EA6', alpha=0.88)
for bars in [b1,b2,b3,b4]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x()+bar.get_width()/2, h),
                    xytext=(0,3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7)
ax.set_ylim(0, 1.1); ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Sarcasm Detection — Model Comparison', fontsize=13)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('results/plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Comparison chart saved')

# ══ PLOT 3: Training curves for BERTweet ═════════════════════════
log         = bertweet_result['training_log']
ep_list     = [e['epoch']      for e in log]
tr_loss_log = [e['train_loss'] for e in log]
vl_loss_log = [e['val_loss']   for e in log]
tr_f1_log   = [e['train_f1']   for e in log]
vl_f1_log   = [e['val_f1']     for e in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(ep_list, tr_loss_log, 'o-', label='Train', color='#2E5FA3')
ax1.plot(ep_list, vl_loss_log, 's--', label='Val',  color='#E0783C')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('BERTweet — Loss per Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_list, tr_f1_log, 'o-', label='Train', color='#2E5FA3')
ax2.plot(ep_list, vl_f1_log, 's--', label='Val',  color='#E0783C')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1 Score')
ax2.set_title('BERTweet — F1 per Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/plots/bertweet_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Training curves saved')

# ══ PLOT 4: Ablation bar chart ════════════════════════════════════
abl_names = ['BERT\n(full FT)', 'RoBERTa\n(full FT)', 'BERTweet\n(full FT)', 'BERTweet\n(frozen)']
abl_f1    = [r['test_metrics']['f1'] for r in ALL_RESULTS]
abl_acc   = [r['test_metrics']['accuracy'] for r in ALL_RESULTS]
colors    = ['#5B8DB8','#5B8DB8','#2E5FA3','#AAC4DE']

x2 = np.arange(len(abl_names)); w2 = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x2 - w2/2, abl_f1,  w2, label='F1',       color=colors, alpha=0.9)
b2 = ax.bar(x2 + w2/2, abl_acc, w2, label='Accuracy',  color=colors, alpha=0.6)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)
ax.set_ylim(0, 1.0); ax.set_xticks(x2); ax.set_xticklabels(abl_names, fontsize=10)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Ablation Study — Full FT vs Frozen Encoder', fontsize=12)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('results/plots/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Ablation chart saved')

Best model : bertweet-base
Test F1    : 0.8052
Test Acc   : 0.8278

─────────────────────────────────────────────────────────────────
ERROR ANALYSIS — bertweet-base
─────────────────────────────────────────────────────────────────
Total test samples : 784
True Positives     : 279
False Positives    : 103  (literal → predicted sarcastic)
False Negatives    : 32  (sarcastic → predicted literal)

── False Positives (literal tweets called sarcastic) ─────────
  [ 1] Corny jokes are my absolute favorite
  [ 2] Most important thing I've learned in school HTTPURL
  [ 3] @USER Not Long To Go Before Main Stream Media Pitch. Surgery follows with me using Prototype demonst
  [ 4] Noo again it's on!!! Pick ANew Song Cant Stand It
  [ 5] @USER No sugar during christmas time? :(
  [ 6] I wonder what was the holiday rituals for true Africans
  [ 7] Pips who drop out of school n have Kirya as 'eir role model tht ey wana sing too..guy spix Portugues
  [ 8] About as relevant as Michael Sam and Jason Col

In [13]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 ─ Final results table
# ═══════════════════════════════════════════════════════════════════
print('=' * 72)
print('FINAL RESULTS TABLE )')
print('=' * 72)
print(f'{"Model":<38} {"Accuracy":>9} {"Precision":>10} {"Recall":>7} {"F1":>7}')
print('-' * 72)

all_rows_final = BASELINE_RESULTS + [
    {'name': r['name'], **r['test_metrics']} for r in ALL_RESULTS
]
for r in all_rows_final:
    print(f"{r['name']:<38} {r['accuracy']:>9.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>7.4f} {r['f1']:>7.4f}")

print('=' * 72)
best = max(all_rows_final, key=lambda r: r['f1'])
print(f'\n✓ Best model: {best["name"]} → F1 = {best["f1"]:.4f}')

# ── Save JSON ─────────────────────────────────────────────────────
os.makedirs('results', exist_ok=True)
with open('results/all_results.json', 'w') as f:
    json.dump({
        'baselines':    BASELINE_RESULTS,
        'transformers': [
            {k: v for k, v in r.items()
             if k not in ('test_true', 'test_preds', 'test_texts')}
            for r in ALL_RESULTS
        ],
    }, f, indent=2)
print('✓ Results saved → results/all_results.json')
print('✓ Plots saved   → results/plots/')

FINAL RESULTS TABLE )
Model                                   Accuracy  Precision  Recall      F1
------------------------------------------------------------------------
Majority Class                            0.3967     0.3967  1.0000  0.5680
TF-IDF + Logistic Regression              0.6633     0.5697  0.6174  0.5926
TF-IDF + LinearSVC                        0.6429     0.5455  0.5981  0.5706
bert-base-uncased                         0.6505     0.5433  0.7460  0.6287
roberta-base                              0.7564     0.6523  0.8264  0.7291
bertweet-base                             0.8278     0.7304  0.8971  0.8052
bertweet-base (frozen)                    0.5077     0.4352  0.8103  0.5663

✓ Best model: bertweet-base → F1 = 0.8052
✓ Results saved → results/all_results.json
✓ Plots saved   → results/plots/


In [14]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 ─ Download everything as a zip
# ═══════════════════════════════════════════════════════════════════
import shutil
from google.colab import files

shutil.make_archive('sarcasm_results', 'zip', 'results')
files.download('sarcasm_results.zip')
print('✓ sarcasm_results.zip downloaded')
print('  Contains: all_results.json + confusion_matrix.png')
print('            + model_comparison.png + bertweet_training_curve.png')
print('            + ablation_study.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ sarcasm_results.zip downloaded
  Contains: all_results.json + confusion_matrix.png
            + model_comparison.png + bertweet_training_curve.png
            + ablation_study.png
